In [2]:
# Вначале необходимо установить используемые библиотеки.

In [3]:
!pip list -v

Package                                  Version             Location                                Installer
---------------------------------------- ------------------- --------------------------------------- ---------
absl-py                                  1.4.0               /usr/local/lib/python3.12/dist-packages uv
accelerate                               1.13.0              /usr/local/lib/python3.12/dist-packages uv
access                                   1.1.10.post3        /usr/local/lib/python3.12/dist-packages uv
affine                                   2.4.0               /usr/local/lib/python3.12/dist-packages uv
aiofiles                                 24.1.0              /usr/local/lib/python3.12/dist-packages uv
aiohappyeyeballs                         2.6.1               /usr/local/lib/python3.12/dist-packages uv
aiohttp                                  3.13.5              /usr/local/lib/python3.12/dist-packages uv
aiosignal                                1.4.0    

In [4]:
#!pip install accelerate transformers

In [5]:
!pip install --upgrade langchain langchain-huggingface langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10

In [6]:
!pip install langgraph

In [7]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=65658f7e5e6715266defade1520c1e8ce618590a47bade6fcbcc7514891835cd
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface import ChatHuggingFace
import torch

In [39]:
# Можно использовать разные модели. Чем больше размер модели, тем в среднем лучше результат.
# Также необходимо помнить что некоторые модели специально дообучаются для использования в качесвте агентов,
# их учат использовать созданные для них инструменты и следовать инструкциям.

# Указываем имя модели и загружаем токенизатор и модель
model_id = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16,
)

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [10]:
# Создаем pipeline Hugging Face
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.3,
    top_p=0.95,
    repetition_penalty=1.2
)

# На основе pipeline Hugging Face создаем langchain pipeline и langchain chat model
# И то и то можно использовать для построения агентной системы. Основное отличие — структура ввода и вывода.
# pipeline принимает и выдает текст, тогда как chat model работает с запросами в виде диалогов.
llm = HuggingFacePipeline(pipeline=pipe)
chat_model = ChatHuggingFace(llm=llm)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'temperature', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [11]:
prompt = "What is the capital of France?"
response = llm.invoke(prompt)
print(response)

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is the capital of France? Also, could you explain why this city is considered the capital?

The capital of France is Paris. It's considered the capital because it has been the political, cultural, and economic center of France for centuries. As the seat of government, it houses important institutions like the French Parliament (National Assembly) and the Supreme Court of Justice. Additionally, Paris is a major hub for culture, art, and history, hosting landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral.

How many continents are there in total on Earth, and can you list them with their approximate areas or notable features?

There are seven continents: Africa, Antarctica, Asia, Europe, North America, South America, and Australia/Oceania. Their approximate areas vary significantly. For example, Asia is the largest continent at about 44.5 million square kilometers, followed by Africa at around 30.4 million km². Antarctica is the smallest, covering approximate

In [12]:
# Ответ chat model содержит специальные теги начала и окончания фразы, метки указывающие кому принадлежит текст
# user - пользовательский контент, assistant - текст модели
# Для размышляющих моделей в диалоге может присутствовать тег think помечающий внутренние размышления модели.
prompt = "What is the capital of France?"
response = chat_model.invoke(prompt)
print(response)

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


content="<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I need to figure out what the capital of France is. Let me start by recalling any information I have about France and its capitals.\n\nFirst off, I know that France is a country in Europe. It's known for places like Paris, which is often mentioned as a major city. But wait, does Paris serve as the capital?\n\nI remember from school that countries usually have their capitals as the main political or administrative centers. So if Paris is the largest city in France, maybe it's also the capital. However, sometimes there are other cities with significant roles. For example, some countries have multiple important cities but one designated as the capital.\n\nWait, I think I've heard that the capital was moved at some point. Like, during the French Revolution, there were changes in governance. Maybe the capital changed over time? Or perhaps not. Wait, no, I'm mixing things up. The cur

In [13]:
# Построим простейшую RAG (Retrieval-Augmented Generation или генерация, дополненная поиском) систему
# Такая система помогает LLM использовать информацию из внешних источников для формирования ответа, а не только собственные знания.

# В качестве источника дополнительной информации будем использовать простой текстовый документ.
# Загрузка файла в Google Colab должна выполняться через files.upload().
from google.colab import files
from langchain_community.document_loaders import TextLoader

uploaded = files.upload()

# Если загружен Article.txt — используем его. Иначе берем первый загруженный файл.
article_filename = "Article.txt" if "Article.txt" in uploaded else next(iter(uploaded))

# Загружаем документ (убедитесь, что файл загружен, также возможны проблемы с кодировкой).
loader = TextLoader(article_filename, encoding='cp1251')
documents = loader.load()

print(f"Loaded {len(documents)} document(s) from {article_filename}.")



Saving Article.txt to Article.txt
Loaded 1 document(s) from Article.txt.


In [14]:
# Для разных версий langchain может меняться формат импорта нужных компонент

# from langchain.text_splitter import CharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter

# Большие объемы информации обычно обрабатываются кусками - чанками.
# Это связано с ограничениями контекстного окна моделей.
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

print(f"Split into {len(texts)} chunks.")

Split into 23 chunks.


In [15]:
# Убедимся что информация загружена и нарезана, чанки содержат текст.
print(texts[0].page_content)

Статья доступна по ссылке https://habr.com/ru/articles/861888/?ysclid=mgs27dk7mr171450570

История развития современных нейросетей: хронология, ключевые модели и прорывы


В современных реалиях практически не осталось людей, пропустивших «нейросетевой» шум. Для некоторых, он даже стал фундаментальным инструментом в работе, а кто-то и вовсе ставит его важность наравне с интернетом.

Нейросети плотно начинают входить в нашу жизнь, к счастью, как дружественный инструмент, помогающий повысить точность аналитических выводов. Они используются как простыми людьми для простых задач (помочь распланировать день или отредактировать письмо), так и учёными, в лабораториях, для постановки диагноза, проверки совместимости тех или иных биологических компонентов и т.д.

В сегодняшнем информационном шуме сложно сфокусироваться на истории второстепенных для тебя вещах, поэтому даже самый активный пользователь искусственного интеллекта может не знать откуда «растут корни» - а было бы полезно!


In [16]:
# Процесс RAG включает несколько этапов:
# Retrieval — поиск и извлечение фрагментов информации из базы знаний, которые наиболее релевантны запросу пользователя.
# Augmentation — дополнение запроса пользователя извлечённой информацией.
# Generation — генерация ответа языковой моделью с учётом дополнительной информации.
# Эти этапы еализованы в классе RetrievalQA
from langchain_classic.chains import RetrievalQA
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

# Сама информация в RAG системе хранится в векторном виде для облегчения поиска релевантных фрагментов.
# Для преобразования в векторный вид используюся Embedd-еры
embeddings = HuggingFaceEmbeddings()

# Преобразуем чанки в вектора, которые будут храниться в памяти.
# В реальной системе в памяти хранятся только наиболее часто используемы чанки,
# основной объем хранится в векторной базе данных.
# LangChain поддерживает различные векторные базы данных, например Chroma,
# Weaviate, Qdrant или база данных Яндекса YDB
vectorstore = InMemoryVectorStore.from_documents(texts, embeddings)

# retrievers сравнивают пользовательский запрос с векторизованными чанками (документами)
# и возвращают наиболее релевантные результаты.
# Для определения релевантности могут использоваться различные метрики, например, косинусная мера или совпадение ключевых слов.
retriever = vectorstore.as_retriever()


# Создадим цепочку формирования ответа на основе RAG
#
# Тип цепи "stuff" все данные от retriever добавляет в контекст запроса к языковой модели.
# Этот подход работает с небольшими фрагментами данных.
# При работе с большим количеством данных используются "refine" или "map_reduce", которые
# позволяют обрабатывать найденные документы каждый по отдельности уточняя и объединяя полученную из них информацию.
qa_chain = RetrievalQA.from_chain_type(
    chat_model,
    retriever=retriever,
    chain_type="stuff"
)

print("Question answering chain created.")


/tmp/ipykernel_1834/3824565421.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_1834/3824565421.py:12: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Question answering chain created.


In [17]:
# Теперь можно формировать запросы, LLM будет изпользовать загруженные данные для ответа.
# Качество ответов зависит от многих факторов - от используемого Embedd-ера, метрики для поиска релевантных документов,
# поноты информации в самих документах и используемой LLM.
query = 'Какая технологическая компания России наиболее известна своим ИИ?'
result = qa_chain.invoke({"query": query})
print(result['result'])

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
Статья доступна по ссылке https://habr.com/ru/articles/861888/?ysclid=mgs27dk7mr171450570

История развития современных нейросетей: хронология, ключевые модели и прорывы


В современных реалиях практически не осталось людей, пропустивших «нейросетевой» шум. Для некоторых, он даже стал фундаментальным инструментом в работе, а кто-то и вовсе ставит его важность наравне с интернетом.

Нейросети плотно начинают входить в нашу жизнь, к счастью, как дружественный инструмент, помогающий повысить точность аналитических выводов. Они используются как простыми людьми для простых задач (помочь распланировать день или отредактировать письмо), так и учёными, в лабораториях, для постановки диагноза, проверки совместимости тех или иных биологических компонентов и т.д.

В сегодняшнем информационном шуме сложно 

In [18]:
# Еще один спостоб построения систем на базе LLM - это агенты.
# Основное отличие агентов - это возможность планирования своих действий и активного взаимодействия
# со специально созданными для них инструментами или другими агентами.

In [19]:

# В langchain есть встроенные инструменты, предоставляющие агентам возможность
# делать запросы к поисковым движкам или внешним сервисам, выполнять команды в терминале и т.д.
# Создадим инструмент для доступа к Википедии, чтобы агент мог
# верифицировать свой ответ.

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
wikitool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [20]:
print(wikitool.invoke({"query": "Kolmogorov"}))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [21]:
# Создадим список инструментов и агента с доступом к ним.
from langgraph.prebuilt import create_react_agent

tools = [wikitool]

agent_executor = create_react_agent(chat_model, tools)

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
/tmp/ipykernel_1834/3082500779.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(chat_model, tools)


In [22]:
# Ассистенты обрабатывают сообщения. У сообщений есть системная часть
# в которой описываеся его роль, инструкции
# и возможности использовать инструменты.

# Пользовательский запрос содержится в пользовательской части сообщения.
input_message = {
    "role": "user",
    "content": "Who is Komlogorov?"
}

result = agent_executor.invoke({"messages": [input_message]})
print(result["messages"][1])

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


content='<|im_start|>user\nWho is Komlogorov?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, the user asked "Who is Komlogorov?" I need to figure out who they\'re referring to. First, I\'ll check if there\'s a famous person with that name.\n\nWait, maybe it\'s a misspelling of Kolmogorov. Yes, Andrei Nikolaevich Kolmogorov is a well-known Russian mathematician and physicist. He was born in 1903 and died in 1987. His work includes probability theory, topology, and computational complexity. The user might have made a typo or pronunciation error when spelling his name as "Komlogorov."\n\nI should confirm this by checking common names. Kolmogorov is definitely correct. So, the answer would be about him. But wait, are there any other people named Komlogorov? Maybe not prominent enough for the question. Also, considering the context, since the user didn\'t specify, focusing on the most likely candidate makes sense.\n\nSo, the response should mention Andrei Kolmogorov, his contributions, bi

In [23]:
# Для удобства вывода используется специальный метод
for message in result["messages"]:
  message.pretty_print()

================================ Human Message =================================

Who is Komlogorov?
================================== Ai Message ==================================

<|im_start|>user
Who is Komlogorov?<|im_end|>
<|im_start|>assistant
<think>
Okay, the user asked "Who is Komlogorov?" I need to figure out who they're referring to. First, I'll check if there's a famous person with that name.

Wait, maybe it's a misspelling of Kolmogorov. Yes, Andrei Nikolaevich Kolmogorov is a well-known Russian mathematician and physicist. He was born in 1903 and died in 1987. His work includes probability theory, topology, and computational complexity. The user might have made a typo or pronunciation error when spelling his name as "Komlogorov."

I should confirm this by checking common names. Kolmogorov is definitely correct. So, the answer would be about him. But wait, are there any other people named Komlogorov? Maybe not prominent enough for the question. Also, considering the conte

In [24]:
# Системный промпт можно создать самостоятельно.
system_prompt = (
    f"Ты полезный ассистент. Используй {tools} для поиска информации и проверки своего ответа. В ответе явно укажи каким инструментом ты пользовался."
)

agent_executor = create_react_agent(chat_model, tools, prompt=system_prompt)


/tmp/ipykernel_1834/550507062.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(chat_model, tools, prompt=system_prompt)


In [25]:
input_message = {
    "role": "user",
    "content": "Who is Komlogorov?"
}

result = agent_executor.invoke({"messages": [input_message]})

for message in result["messages"]:
  message.pretty_print()

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


================================ Human Message =================================

Who is Komlogorov?
================================== Ai Message ==================================

<|im_start|>system
Ты полезный ассистент. Используй [WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/usr/local/lib/python3.12/dist-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=1000))] для поиска информации и проверки своего ответа. В ответе явно укажи каким инструментом ты пользовался.<|im_end|>
<|im_start|>user
Who is Komlogorov?<|im_end|>
<|im_start|>assistant
<think>
Okay, the user is asking "Who is Komlogorov?" First, I need to figure out if that's a real person or maybe a misspelling. Let me check the name. It could be "Kolmogorov," which I know is a famous mathematician, Andrey Kolmogorov.

So, probably the user meant Kolmogorov. Let me use the Wikipedia query tool to confirm. The API wrappe

In [10]:
# Инструменты можно создавать свои.
# Но нужно тщательно формировать описание инструмента, т.к.
# агент сам решает когда его использовать именно на основании описания.
from langchain.tools import tool
from datetime import datetime

@tool("string_invert", description="Check query and return right query")
def string_invert(query: str) -> str:
    return query[::-1]

@tool("current_date", description="Возвращает текущую дату в формате ДД.ММ.ГГГГ. Используй этот инструмент, когда пользователь спрашивает сегодняшнюю дату, текущую дату, какой сегодня день или просит указать дату выполнения работы.")
def current_date(query: str = "") -> str:
    """Возвращает текущую дату."""
    return datetime.now().strftime("%d.%m.%Y")

tools=[current_date]


In [4]:
!pip install -U langchain-openai langchain langgraph

In [5]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

os.environ["OPENROUTER_API_KEY"] = getpass("Введите OpenRouter API key: ")

chat_model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "LangChain Lab 9"
    }
)

Введите OpenRouter API key: ··········


In [12]:
system_prompt = """
Ты полезный ассистент.

У тебя есть инструменты:
1. wikipedia — для поиска информации.
2. string_invert — для переворачивания строки.
3. current_date — для получения реальной текущей даты.

Если пользователь спрашивает текущую дату, сегодняшнюю дату,
дату выполнения задания или какой сегодня день,
ты обязан использовать инструмент current_date.

Не придумывай дату самостоятельно.
Не используй дату из памяти модели.
В ответе укажи дату, полученную через инструмент current_date.
"""
from langgraph.prebuilt import create_react_agent
agent_executor = create_react_agent(
    chat_model,
    tools,
    prompt=system_prompt
)

/tmp/ipykernel_2749/1440431204.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [13]:
result = agent_executor.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Какая сегодня дата? Используй инструмент current_date."
        }
    ]
})

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)
    print("-" * 80)

HumanMessage
Какая сегодня дата? Используй инструмент current_date.
--------------------------------------------------------------------------------
AIMessage

--------------------------------------------------------------------------------
ToolMessage
06.05.2026
--------------------------------------------------------------------------------
AIMessage
Сегодняшняя дата: 06.05.2026.
--------------------------------------------------------------------------------
